In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from relbench.datasets import get_dataset
from relbench.tasks import get_task

%load_ext autoreload
%autoreload 2

dataset = get_dataset(name='rel-trial', process=False)

In [ ]:
dataset.get_db().table_dict.keys()

In [ ]:
for i,j in dataset.get_db().table_dict.items():
    print(i)
    display(j.df.head(1))

# Task 1: trial-outcome: if a trial will meet its primary outcome?

In [ ]:
task = get_task('rel-trial', 'study-outcome', download=True)

In [ ]:
task.get_table("train").df.columns

In [ ]:
print('TRAIN\n')

print(f'{len(task.get_table("train").df):,} samples over {task.get_table("train").df.nct_id.nunique():,} trials.')
print()
print('Label Distribution:')
print(task.get_table("train").df.outcome.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(task.get_table("train").df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of trials (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    task.get_table("train").df
    .drop_duplicates(subset='nct_id', keep='last')
    .groupby('timestamp').nct_id.nunique() / task.get_table("train").df.nct_id.nunique()
)

In [ ]:
print('VAL\n')

print(f'{len(task.get_table("val").df):,} samples over {task.get_table("val").df.nct_id.nunique():,} trials.')
print()
print('Label Distribution:')
print(task.get_table("val").df.outcome.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(task.get_table("val").df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print(f'User overlap between train and val: {len(set(task.get_table("train").df.nct_id) & set(task.get_table("val").df.nct_id)):,}')

Each validation/test trials are completely new trials. So it is almost like an inductive setting - a new node comes in to the graph

# Task 2: trial-adverse: what is number of adverse events?

In [ ]:
task = get_task('rel-trial', 'study-adverse', download=True)

In [ ]:
sns.boxplot(task.get_table("train").df.num_of_adverse_events.values, showfliers=False)

In [ ]:
sns.boxplot(task.get_table("val").df.num_of_adverse_events.values, showfliers=False)

In [ ]:
task.get_table("train").df.num_of_adverse_events.value_counts()

In [ ]:
len(task.get_table("train").df[task.get_table("train").df.num_of_adverse_events == 0])/len(task.get_table("train").df)

40.8% of training trials have 0 adverse events

In [ ]:
len(task.get_table("val").df[task.get_table("val").df.num_of_adverse_events == 0])/len(task.get_table("val").df)

In [ ]:
sns.distplot(task.get_table("val").df.num_of_adverse_events)

# Task 3: trial-withdrawl: Predict the the set of reasons of withdrawals for each trial in the next 1 year

In [ ]:
task = get_task('rel-trial', 'study-withdrawal', download=True)

In [ ]:
import numpy as np
## check label imbalance
np.sum(np.vstack(task.get_table("train").df.withdraw_reasons.values), axis = 0)

In [ ]:
## check how many labels do each trial have
sns.histplot(np.sum(np.vstack(task.get_table("train").df.withdraw_reasons.values), axis = 1))

In [ ]:
len(task.get_table("train").df.nct_id.unique())

In [ ]:
len(task.get_table("val").df.nct_id.unique())

In [ ]:
# check if any of the label is correlated to each other
y = np.vstack(task.get_table("train").df.withdraw_reasons.values)

In [ ]:
corr = pd.DataFrame(y).corr()

In [ ]:
np.fill_diagonal(corr.values, np.nan)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Label Correlation Heatmap')
plt.show()

# Task 4: rel-trial-site: Predict the success rate of a trial site in the next 1 year.

In [ ]:
task = get_task('rel-trial', 'site-success', download=True)

In [ ]:
sns.distplot(task.get_table("train").df.success_rate)
sns.distplot(task.get_table("val").df.success_rate)